# 🐍 Colab でここまでできる！ Python 作品ギャラリー

Python と Google Colab を使うと、**ブラウザだけ**で次のようなプログラムが作れます。
授業で学ぶ `変数・if・for・リスト・関数` が、こうした高度なプログラムの土台になっています。

| No. | 作品 | 分野 |
|---|---|---|
| 1 | 🎵 プログラムで音楽を作る | 音・デジタル化 |
| 2 | 🌀 フラクタル図形を描く | 数学・コンピュータグラフィックス |
| 3 | 🎯 乱数で円周率を求める | シミュレーション |
| 4 | 🦠 感染症の広がりを予測する | モデル化とシミュレーション |
| 5 | 🚀 ボール投げの軌道をスライダーで操作 | 物理・インタラクティブ |
| 6 | 🧬 ライフゲームのアニメーション | アルゴリズム・アニメーション |
| 7 | 🌸 データ分析（花のデータ） | データの活用 |
| 8 | ☀️ Web から天気予報を取得してグラフ化 | ネットワーク・API |
| 9 | 🤖 AI が手書き数字を認識する | 機械学習 |
| 10 | 🖼️ AI で写真の色を減らす（画像処理） | 画像のデジタル化・機械学習 |
| 11 | 🌐 マウスで回せる 3D グラフ | 可視化 |

## ▶️ 使い方
- メニューの **「ランタイム」→「すべてのセルを実行」** で全部動きます（1〜2 分）
- 各セルは **Shift + Enter** で 1 つずつ実行することもできます
- 右側に入力欄やスライダーが出るセルは、値を変えてもう一度実行すると結果が変わります
- 最初に **「ファイル」→「ドライブにコピーを保存」** すると、自由に書き換えて試せます

> ℹ️ グラフ中の文字は、Colab に日本語フォントが入っていないため英語で表示しています。

In [ ]:
# @title 準備：使うライブラリを読み込む（最初に 1 回実行）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, HTML, display

plt.rcParams["figure.dpi"] = 100
print("準備完了！")

---
# 1. 🎵 プログラムで音楽を作る
音は空気の振動＝**波**です。サイン波の数値を計算して並べると、コンピュータで音が作れます。下のプレーヤーで ▶ を押すと「きらきら星」が流れます。

| 使っている技術 | 授業とのつながり |
|---|---|
| `numpy`（数値計算）、`IPython.display.Audio` | 第7回 リスト・第8回 関数／情報I「音のデジタル化」 |

In [ ]:
# @title きらきら星を演奏する
tempo = 120  # @param {type:"slider", min:60, max:240, step:10}
wave_type = "sine"  # @param ["sine", "square"]

RATE = 44100                       # 1秒あたりのサンプル数（サンプリング周波数）
FREQ = {"ド": 262, "レ": 294, "ミ": 330, "ファ": 349, "ソ": 392, "ラ": 440, "シ": 494}

def make_tone(freq, sec):
    """周波数 freq [Hz] の音を sec 秒ぶん作る"""
    t = np.linspace(0, sec, int(RATE * sec), endpoint=False)
    wave = np.sin(2 * np.pi * freq * t)
    if wave_type == "square":
        wave = np.sign(wave)          # 四角い波にするとゲーム機のような音に
    fade = np.minimum(1, (sec - t) * 20)   # 音の終わりをなめらかに
    return wave * fade * 0.3

melody = "ド ド ソ ソ ラ ラ ソ_ ファ ファ ミ ミ レ レ ド_".split()
beat = 60 / tempo
song = []
for note in melody:
    length = beat * 2 if note.endswith("_") else beat   # _ がついた音は 2 拍
    song.append(make_tone(FREQ[note.rstrip("_")], length))
song = np.concatenate(song)

display(Audio(song, rate=RATE))

# 最初の 0.02 秒ぶんの波の形を見てみる
plt.figure(figsize=(8, 2.5))
plt.plot(song[: int(RATE * 0.02)])
plt.title("Waveform of the first 0.02 seconds")
plt.xlabel("sample"); plt.ylabel("amplitude")
plt.show()

> 🧪 **やってみよう**：`melody` を書き換えて別の曲にしてみよう。`wave_type` を `square` にすると音色が変わります。

---
# 2. 🌀 フラクタル図形（マンデルブロ集合）を描く
「$z \\leftarrow z^2 + c$」という簡単な計算を繰り返すだけで、拡大しても同じような模様が無限に現れる不思議な図形が描けます。数十万個の点の計算を、numpy で一気に行います。

| 使っている技術 | 授業とのつながり |
|---|---|
| `numpy` の配列計算、`matplotlib` | 第5回 繰り返し・第4回 条件分岐 |

In [ ]:
# @title マンデルブロ集合（中心と拡大率を変えられる）
center_x = -0.6  # @param {type:"number"}
center_y = 0.0  # @param {type:"number"}
zoom = 1  # @param {type:"slider", min:1, max:500, step:1}
max_iter = 200

w, h = 900, 600
scale = 1.5 / zoom
x = np.linspace(center_x - scale * 1.5, center_x + scale * 1.5, w)
y = np.linspace(center_y - scale, center_y + scale, h)
c = x[np.newaxis, :] + 1j * y[:, np.newaxis]   # 複素数の格子
z = np.zeros_like(c)
count = np.zeros(c.shape, dtype=int)

for i in range(max_iter):
    inside = np.abs(z) <= 2          # まだ発散していない点だけ
    z[inside] = z[inside] ** 2 + c[inside]
    count[inside] += 1

plt.figure(figsize=(9, 6))
plt.imshow(count, cmap="twilight_shifted", extent=[x[0], x[-1], y[0], y[-1]], origin="lower")
plt.title(f"Mandelbrot set  (zoom x{zoom})")
plt.axis("off")
plt.show()

> 🧪 **やってみよう**：`center_x = -0.745, center_y = 0.11, zoom = 30` にすると「タツノオトシゴの谷」と呼ばれる部分が拡大されます。さらに `zoom` を 300 にしてみよう。

---
# 3. 🎯 乱数で円周率を求める（モンテカルロ法）
正方形の中にランダムに点を打ち、円の中に入った割合から円周率 π を求めます。点を増やすほど 3.14159… に近づいていく様子がわかります。

| 使っている技術 | 授業とのつながり |
|---|---|
| `numpy.random`、散布図・折れ線グラフ | 第9回 ライブラリ（random）／情報I「シミュレーション」 |

In [ ]:
# @title 点を打って円周率を推定する
n_points = 5000  # @param {type:"slider", min:100, max:50000, step:100}

rng = np.random.default_rng()
px = rng.random(n_points)
py = rng.random(n_points)
inside = px ** 2 + py ** 2 <= 1           # 円の中に入ったか
pi_est = 4 * inside.sum() / n_points

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].scatter(px[inside], py[inside], s=2, color="tab:red", label="inside")
ax[0].scatter(px[~inside], py[~inside], s=2, color="tab:blue", label="outside")
ax[0].set_aspect("equal"); ax[0].legend(loc="upper right")
ax[0].set_title(f"{n_points} points  ->  pi = {pi_est:.4f}")

running = 4 * np.cumsum(inside) / np.arange(1, n_points + 1)   # 途中経過
ax[1].plot(np.arange(1, n_points + 1), running)
ax[1].axhline(np.pi, color="red", linestyle="--", label="true pi")
ax[1].set_xscale("log"); ax[1].set_ylim(2.6, 3.6)
ax[1].set_xlabel("number of points"); ax[1].set_title("Estimate approaches pi")
ax[1].legend()
plt.show()
print(f"推定値: {pi_est}   本当の円周率: {np.pi:.6f}   誤差: {abs(pi_est - np.pi):.4f}")

> 🧪 **やってみよう**：`n_points` を増やすと精度はどう変わるか。何回か実行して結果がばらつくことも確かめよう。

---
# 4. 🦠 感染症の広がりを予測する（SIR モデル）
人を「感染していない人(S)」「感染している人(I)」「回復した人(R)」に分け、1 日ごとの変化を計算します。**外出自粛などで接触を減らすと、ピークがどう変わるか** を比べられます。

| 使っている技術 | 授業とのつながり |
|---|---|
| 数理モデル、`for` 文による時間発展の計算 | 第5回 繰り返し・第7回 リスト／情報I「モデル化とシミュレーション」 |

In [ ]:
# @title 接触の減らし方を変えて比べる
population = 10000  # @param {type:"integer"}
beta = 0.3  # @param {type:"slider", min:0.05, max:0.6, step:0.01}
recovery_days = 7  # @param {type:"slider", min:2, max:20, step:1}
contact_reduction = 40  # @param {type:"slider", min:0, max:90, step:5}

def simulate(beta, days=160):
    """SIRモデルを1日ずつ計算して、S, I, R のリストを返す"""
    gamma = 1 / recovery_days
    S, I, R = [population - 10], [10], [0]
    for d in range(days):
        new_inf = beta * S[-1] * I[-1] / population
        new_rec = gamma * I[-1]
        S.append(S[-1] - new_inf)
        I.append(I[-1] + new_inf - new_rec)
        R.append(R[-1] + new_rec)
    return S, I, R

S1, I1, R1 = simulate(beta)
S2, I2, R2 = simulate(beta * (1 - contact_reduction / 100))

plt.figure(figsize=(10, 5))
plt.plot(I1, color="tab:red", label="Infected (no measures)")
plt.plot(I2, color="tab:red", linestyle="--", label=f"Infected ({contact_reduction}% less contact)")
plt.plot(R1, color="tab:green", alpha=0.5, label="Recovered (no measures)")
plt.plot(R2, color="tab:green", alpha=0.5, linestyle="--", label=f"Recovered ({contact_reduction}% less contact)")
plt.xlabel("day"); plt.ylabel("people"); plt.title("SIR model simulation")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f"対策なし : 感染者のピーク {max(I1):6.0f} 人（{I1.index(max(I1))}日目）")
print(f"対策あり : 感染者のピーク {max(I2):6.0f} 人（{I2.index(max(I2))}日目）")

> 🧪 **やってみよう**：`contact_reduction` を 0〜90 で変えてみよう。ピークの高さと時期はどう変わるだろう？ 医療の観点から何が大切かも考えてみよう。

---
# 5. 🚀 ボール投げの軌道をスライダーで操作する
投げる角度と速さを **スライダーで動かすと、その場でグラフが更新** されます。何度で投げると一番遠くまで届くか、実験してみましょう。

| 使っている技術 | 授業とのつながり |
|---|---|
| `ipywidgets`（スライダー部品）、物理の公式 | 第8回 関数／物理「放物運動」 |

In [ ]:
# @title スライダーを動かしてみよう
from ipywidgets import interact, IntSlider

G = 9.8

def throw(angle=45, speed=20):
    rad = np.radians(angle)
    t_end = 2 * speed * np.sin(rad) / G          # 地面に落ちるまでの時間
    t = np.linspace(0, t_end, 200)
    x = speed * np.cos(rad) * t
    y = speed * np.sin(rad) * t - G * t ** 2 / 2

    plt.figure(figsize=(9, 4))
    plt.plot(x, y, linewidth=3)
    plt.xlim(0, 105); plt.ylim(0, 55)
    plt.xlabel("distance [m]"); plt.ylabel("height [m]")
    plt.title(f"angle {angle} deg, speed {speed} m/s  ->  distance {x[-1]:.1f} m, max height {y.max():.1f} m")
    plt.grid(alpha=0.3)
    plt.show()

interact(throw,
         angle=IntSlider(value=45, min=5, max=85, step=5, description="角度(度)"),
         speed=IntSlider(value=20, min=5, max=32, step=1, description="速さ(m/s)"));

> 🧪 **やってみよう**：一番遠くまで届く角度は何度？ 30度と60度の飛距離を比べると何がわかるかな？

---
# 6. 🧬 ライフゲームのアニメーション
マス目の生き物が「まわりの仲間の数」という簡単なルールだけで生まれたり消えたりします。単純なルールから複雑な動きが生まれる様子を **アニメーション** で見てみましょう。

| 使っている技術 | 授業とのつながり |
|---|---|
| 2次元配列、`matplotlib.animation` | 第4回 条件分岐・第5回 二重ループ |

In [ ]:
# @title ライフゲーム（下の ▶ ボタンで再生）
from matplotlib import animation

size = 60  # @param {type:"slider", min:20, max:100, step:10}
density = 0.25  # @param {type:"slider", min:0.05, max:0.6, step:0.05}
frames = 80

rng = np.random.default_rng()
board = (rng.random((size, size)) < density).astype(int)

def step(b):
    """ルール：生きているマスは周り2〜3匹なら生存、空きマスは周りがちょうど3匹なら誕生"""
    neighbors = sum(np.roll(np.roll(b, dy, axis=0), dx, axis=1)
                    for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0))
    return ((neighbors == 3) | ((b == 1) & (neighbors == 2))).astype(int)

fig, ax = plt.subplots(figsize=(5, 5))
img = ax.imshow(board, cmap="Greens", vmin=0, vmax=1)
ax.axis("off")

def update(i):
    global board
    board = step(board)
    img.set_data(board)
    ax.set_title(f"generation {i + 1}")
    return [img]

anim = animation.FuncAnimation(fig, update, frames=frames, interval=120)
plt.close(fig)
display(HTML(anim.to_jshtml()))

> 🧪 **やってみよう**：`density` を変えて、生き物がすぐ全滅する場合・増え続ける場合を探してみよう。

---
# 7. 🌸 データ分析：花のデータから特徴を見つける
アヤメ（iris）という花 150 本の、花びらやがくの長さのデータを分析します。表の集計とグラフ化で、**品種ごとの特徴** が一目でわかります。

| 使っている技術 | 授業とのつながり |
|---|---|
| `pandas`（表計算）、`scikit-learn` 付属のデータセット | 第7回 リスト／情報I「データの活用」 |

In [ ]:
# @title データを読み込んで集計・可視化する
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df = df.drop(columns="target")

print("▼ データの先頭5行")
display(df.head())
print("▼ 品種ごとの平均値")
display(df.groupby("species").mean().round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, g in df.groupby("species"):
    ax[0].scatter(g["petal length (cm)"], g["petal width (cm)"], label=name, alpha=0.7)
ax[0].set_xlabel("petal length (cm)"); ax[0].set_ylabel("petal width (cm)")
ax[0].set_title("Petal size by species"); ax[0].legend()

df.boxplot(column="sepal length (cm)", by="species", ax=ax[1])
ax[1].set_title("Sepal length by species"); ax[1].set_xlabel("")
plt.suptitle("")
plt.show()

> 🧪 **やってみよう**：グラフから、品種を見分けるにはどの長さに注目すればよいか考えよう。次の「9. AI」はこの考え方をコンピュータに学ばせるものです。

---
# 8. ☀️ Web から天気予報を取得してグラフにする
インターネット上の天気予報サービス（Open-Meteo の **API**）にプログラムからアクセスし、最新の 7 日間予報を取得してグラフにします。

| 使っている技術 | 授業とのつながり |
|---|---|
| `requests`（Web 通信）、JSON データ | 情報I「ネットワーク」「データの活用」 |

In [ ]:
# @title 好きな都市の天気予報を取得
city = "東京"  # @param ["東京", "大阪", "札幌", "福岡", "那覇"]
CITIES = {"東京": (35.68, 139.76), "大阪": (34.69, 135.50), "札幌": (43.06, 141.35),
          "福岡": (33.59, 130.40), "那覇": (26.21, 127.68)}

import requests
lat, lon = CITIES[city]
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": lat, "longitude": lon, "timezone": "Asia/Tokyo",
          "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max"}
try:
    data = requests.get(url, params=params, timeout=10).json()["daily"]
    print(f"✅ {city} の最新の予報を取得しました")
except Exception as e:
    print("⚠️ 通信できなかったので、サンプルデータで表示します:", e)
    data = {"time": [f"day{i}" for i in range(1, 8)],
            "temperature_2m_max": [18, 20, 21, 17, 16, 19, 22],
            "temperature_2m_min": [10, 11, 13, 12, 9, 10, 12],
            "precipitation_probability_max": [10, 0, 30, 80, 60, 20, 10]}

weather = pd.DataFrame(data).rename(columns={
    "time": "日付", "temperature_2m_max": "最高気温", "temperature_2m_min": "最低気温",
    "precipitation_probability_max": "降水確率"})
display(weather)

fig, ax1 = plt.subplots(figsize=(10, 4.5))
days = [d[5:] if len(d) == 10 else d for d in weather["日付"]]
ax2 = ax1.twinx()
ax2.bar(days, weather["降水確率"], color="skyblue", alpha=0.5)
ax2.set_ylim(0, 100); ax2.set_ylabel("rain probability [%]")
ax1.plot(days, weather["最高気温"], "o-", color="tab:red", label="max temp")
ax1.plot(days, weather["最低気温"], "o-", color="tab:blue", label="min temp")
ax1.set_ylabel("temperature [C]"); ax1.legend(loc="upper left")
ax1.set_zorder(ax2.get_zorder() + 1); ax1.patch.set_visible(False)
plt.title("7-day weather forecast")
plt.show()

> 🧪 **やってみよう**：`city` を変えて、地域による気温の違いを比べてみよう。

---
# 9. 🤖 AI が手書き数字を認識する（機械学習）
手書き数字の画像 約1800 枚のうち、**8 割で AI に学習させ、残り 2 割のテスト** をします。人がルールを書かなくても、データから見分け方を学んで高い正解率を出します。

| 使っている技術 | 授業とのつながり |
|---|---|
| `scikit-learn`（機械学習）、サポートベクターマシン | 情報I「データの活用」／AI のしくみ |

In [ ]:
# @title AI に数字を学習させてテストする
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

digits = load_digits()              # 8×8ピクセルの手書き数字画像
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    digits.data, digits.target, digits.images, test_size=0.2, random_state=0)

model = SVC(gamma=0.001)            # AIのモデルを用意して
model.fit(X_train, y_train)         # 学習させる
pred = model.predict(X_test)        # テスト用の画像を予測

print(f"学習に使った画像: {len(X_train)} 枚 / テストした画像: {len(X_test)} 枚")
print(f"🎯 正解率: {accuracy_score(y_test, pred) * 100:.1f} %")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img, p, t in zip(axes.ravel(), img_test, pred, y_test):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(f"AI: {p}", color="green" if p == t else "red")
    ax.axis("off")
plt.suptitle("Test images and AI predictions (red = wrong)")
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title("Confusion matrix (row: true, column: AI)")
plt.show()

> 🧪 **やってみよう**：`test_size` を 0.9 にする（学習データを 1 割に減らす）と正解率はどうなる？ AI にとってのデータ量の大切さを確かめよう。

---
# 10. 🖼️ AI で写真の色を減らす（画像処理）
写真は **ピクセルの色（R, G, B の数値）の集まり** です。似た色をグループ分けする AI（k-means 法）で、何十万色もある写真を少ない色で表現してみます。

| 使っている技術 | 授業とのつながり |
|---|---|
| 画像データ（3次元配列）、`scikit-learn` の k-means 法 | 情報I「画像のデジタル化」 |

In [ ]:
# @title 写真を少ない色で表現する
from sklearn.datasets import load_sample_image
from sklearn.cluster import KMeans

n_colors = 8  # @param {type:"slider", min:2, max:32, step:1}

photo = load_sample_image("china.jpg")          # 縦427×横640×3色 の配列
pixels = photo.reshape(-1, 3) / 255
print(f"画像サイズ: {photo.shape}, 使われている色の数: {len(np.unique(photo.reshape(-1, 3), axis=0)):,} 色")

rng = np.random.default_rng(0)
sample = pixels[rng.choice(len(pixels), 3000, replace=False)]
kmeans = KMeans(n_clusters=n_colors, n_init=4, random_state=0).fit(sample)
reduced = kmeans.cluster_centers_[kmeans.predict(pixels)].reshape(photo.shape)

gray = photo.mean(axis=2)                        # グレースケール
edge = np.abs(np.diff(gray, axis=0))[:, :-1] + np.abs(np.diff(gray, axis=1))[:-1, :]   # 輪郭

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax[0, 0].imshow(photo); ax[0, 0].set_title("Original")
ax[0, 1].imshow(reduced); ax[0, 1].set_title(f"{n_colors} colors (k-means)")
ax[1, 0].imshow(gray, cmap="gray"); ax[1, 0].set_title("Grayscale")
ax[1, 1].imshow(edge, cmap="gray_r", vmax=60); ax[1, 1].set_title("Edge detection")
for a in ax.ravel(): a.axis("off")
plt.show()

plt.figure(figsize=(8, 1))
plt.imshow([kmeans.cluster_centers_]); plt.axis("off"); plt.title("Palette chosen by AI")
plt.show()

> 🧪 **やってみよう**：`n_colors` を 2, 4, 16 と変えて、何色あれば元の写真らしく見えるか調べよう。データ量（情報量）との関係も考えてみよう。

---
# 11. 🌐 マウスで回せる 3D グラフ
`plotly` を使うと、**マウスでドラッグして回転・ホイールで拡大** できる 3D グラフが作れます。水面に広がる波のような曲面を描いてみます。

| 使っている技術 | 授業とのつながり |
|---|---|
| `plotly`（インタラクティブなグラフ） | 数学「関数とグラフ」 |

In [ ]:
# @title 3D グラフ（ドラッグで回転・ホイールで拡大）
import plotly.graph_objects as go

frequency = 1.0  # @param {type:"slider", min:0.5, max:3, step:0.1}

x = np.linspace(-6, 6, 120)
y = np.linspace(-6, 6, 120)
X, Y = np.meshgrid(x, y)
R = np.sqrt(X ** 2 + Y ** 2)
Z = np.sin(frequency * R) / (R + 1) * 3

fig = go.Figure(go.Surface(x=X, y=Y, z=Z, colorscale="Viridis"))
fig.update_layout(title="z = 3 sin(kr) / (r + 1)", width=750, height=600,
                  scene=dict(zaxis=dict(range=[-2, 3])))
fig.show()

> 🧪 **やってみよう**：`Z = ...` の式を `np.sin(X) * np.cos(Y)` など自分の好きな式に変えてみよう。

---
# 🎓 まとめ
ここで見たプログラムは、どれも授業で学ぶ基本の組み合わせでできています。

| 基本の文法 | ここで使われていた場所 |
|---|---|
| 変数・計算 | 音の周波数、物理の公式、円周率の計算 |
| 条件分岐 `if` | ライフゲームのルール、円の内側かどうか |
| 繰り返し `for` | 感染症の 1 日ごとの計算、フラクタルの反復計算 |
| リスト | メロディ、感染者数の記録 |
| 関数 `def` | 音を作る関数、ボールを投げる関数 |
| ライブラリ `import` | numpy・pandas・matplotlib・scikit-learn・plotly |

基本をしっかり身につければ、**音楽・アート・科学・AI・データ分析** など、好きな分野で自分の作品が作れるようになります。

> 💬 Colab のセルにある「生成」ボタン（Gemini）に日本語で「〜するプログラムを書いて」と頼むと、コードの例を作ってもらうこともできます。ただし、**内容を理解して確かめる力** が大切です。